In [16]:
"""
Cellphones.com.vn Mobile Scraper
=================================
Crawl toàn bộ sản phẩm điện thoại từ cellphones.com.vn
Trả về JSON theo cấu trúc database đã định nghĩa.

Chạy: python scraper.py
Output: products.json
"""

import requests
import json
import time
import re
import logging
import random
from bs4 import BeautifulSoup
from dataclasses import dataclass, field, asdict
from typing import Optional
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─── Logging ─────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ─── Config ───────────────────────────────────────────────────────────────────
BASE_URL        = "https://cellphones.com.vn"
MOBILE_URL      = f"{BASE_URL}/mobile.html"
OUTPUT_FILE     = "products.json"
DELAY_MIN       = 1.0   # giây, delay ngẫu nhiên giữa requests
DELAY_MAX       = 2.5
MAX_WORKERS     = 3     # số luồng crawl song song
REQUEST_TIMEOUT = 15

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en;q=0.8",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Referer": BASE_URL,
}

# ─── Data classes (map sang DB entity) ───────────────────────────────────────
@dataclass
class DiscountDTO:
    discount_type:    str   = "FIXED"
    discount_amount:  float = 0
    discount_percent: int   = 0
    is_active:        bool  = True
    start_at:         str   = ""
    end_at:           str   = ""

@dataclass
class VariantDTO:
    color:            str            = ""
    storage_label:    str            = ""
    storage_gb:       Optional[int]  = None
    ram_gb:           Optional[int]  = None
    price:            float          = 0
    compare_at_price: Optional[float]= None
    slug:             str            = ""
    color_image_url:  str            = ""
    sku:              str            = ""
    discount:         Optional[DiscountDTO] = None

@dataclass
class SpecDTO:
    key:        str = ""
    value:      str = ""
    sort_order: int = 0

@dataclass
class ProductDTO:
    name:               str            = ""
    base_name:          str            = ""
    slug:               str            = ""
    short_description:  str            = ""
    detail_description: str            = ""
    thumbnail_url:      str            = ""
    brand_slug:         str            = ""
    category_slug:      str            = "dien-thoai"
    series_slug:        str            = ""
    warranty_months:    int            = 12
    is_featured:        bool           = False
    status:             str            = "ACTIVE"
    sale:               int            = 0
    specs:              list[SpecDTO]  = field(default_factory=list)
    image_urls:         list[str]      = field(default_factory=list)
    variants:           list[VariantDTO] = field(default_factory=list)
    source_url:         str            = ""
    crawled_at:         str            = ""


# ─── Session ─────────────────────────────────────────────────────────────────
def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(HEADERS)
    return s


def fetch(session: requests.Session, url: str) -> Optional[BeautifulSoup]:
    """Fetch URL, trả về BeautifulSoup hoặc None nếu lỗi."""
    try:
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
        res = session.get(url, timeout=REQUEST_TIMEOUT)
        res.raise_for_status()
        return BeautifulSoup(res.text, "lxml")
    except requests.RequestException as e:
        log.error(f"Fetch error [{url}]: {e}")
        return None


# ─── Helpers ─────────────────────────────────────────────────────────────────
def clean_price(text: str) -> float:
    """'33.790.000đ' → 33790000.0"""
    if not text:
        return 0.0
    digits = re.sub(r"[^\d]", "", text)
    return float(digits) if digits else 0.0


def parse_storage_gb(label: str) -> Optional[int]:
    """'256GB' → 256, '1TB' → 1024"""
    if not label:
        return None
    label = label.strip().upper()
    if "TB" in label:
        num = re.sub(r"[^\d]", "", label)
        return int(num) * 1024 if num else None
    if "GB" in label:
        num = re.sub(r"[^\d]", "", label)
        return int(num) if num else None
    return None


def generate_slug(text: str) -> str:
    """'iPhone 17 Pro' → 'iphone-17-pro'"""
    text = text.lower()
    replacements = {
        r"[áàảãạăắặằẳẵâấầẩẫậ]": "a",
        r"[éèẻẽẹêếềểễệ]": "e",
        r"[íìỉĩị]": "i",
        r"[óòỏõọôốồổỗộơớờởỡợ]": "o",
        r"[úùủũụưứừửữự]": "u",
        r"[ýỳỷỹỵ]": "y",
        r"đ": "d",
    }
    for pattern, repl in replacements.items():
        text = re.sub(pattern, repl, text)
    text = re.sub(r"[^a-z0-9\s-]", "", text)
    text = re.sub(r"\s+", "-", text.strip())
    return text


def generate_sku(base_name: str, storage: str, color: str) -> str:
    """'iPhone 17 Pro' + '256GB' + 'Cam Vũ Trụ' → 'IP17P-256GB-CAM'"""
    words   = re.sub(r"[aeiouàáảãạăắặằẳẵâấầẩẫậèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵ\s]",
                     "", base_name, flags=re.IGNORECASE)
    prefix  = words.upper()[:6]
    storage = storage.upper()
    color_c = re.sub(r"\s+", "", color).upper()[:3]
    return f"{prefix}-{storage}-{color_c}"


def make_discount(price: float, compare_at: float) -> Optional[DiscountDTO]:
    """Tạo DiscountDTO nếu có giảm giá."""
    if compare_at and compare_at > price:
        amount  = compare_at - price
        percent = round(amount * 100 / compare_at)
        return DiscountDTO(
            discount_type    = "FIXED",
            discount_amount  = amount,
            discount_percent = percent,
            is_active        = True,
            start_at         = datetime.now().isoformat(timespec="seconds"),
            end_at           = (datetime.now() + timedelta(days=30)).isoformat(timespec="seconds"),
        )
    return None


def detect_brand(name: str) -> str:
    """Detect brand slug từ tên sản phẩm."""
    name_lower = name.lower()
    brands = {
        "apple": ["iphone"],
        "samsung": ["samsung", "galaxy"],
        "xiaomi": ["xiaomi", "redmi", "poco"],
        "oppo": ["oppo"],
        "vivo": ["vivo"],
        "realme": ["realme"],
        "nokia": ["nokia"],
        "motorola": ["motorola", "moto"],
        "oneplus": ["oneplus"],
        "honor": ["honor"],
        "google": ["google", "pixel"],
        "sony": ["sony", "xperia"],
        "asus": ["asus", "zenfone"],
        "huawei": ["huawei"],
    }
    for slug, keywords in brands.items():
        if any(kw in name_lower for kw in keywords):
            return slug
    return "other"


# ─── Step 1: Lấy danh sách URL sản phẩm từ trang danh mục ───────────────────
def get_product_urls(session: requests.Session) -> list[str]:
    """
    Crawl tất cả trang phân trang của /mobile.html
    Trả về list URL sản phẩm.
    """
    urls    = []
    page    = 1
    seen    = set()

    log.info("Bắt đầu lấy danh sách sản phẩm...")

    while True:
        page_url = f"{MOBILE_URL}?page={page}" if page > 1 else MOBILE_URL
        log.info(f"  Đang crawl trang danh mục: {page_url}")

        soup = fetch(session, page_url)
        if not soup:
            break

        # Các selector phổ biến của cellphones.com.vn
        cards = (
            soup.select("a.product-item")
            or soup.select(".product-list-filter a[href*='.html']")
            or soup.select(".cps-product-item a")
            or soup.select("[class*='product'] a[href$='.html']")
        )

        if not cards:
            log.info(f"  Không còn sản phẩm ở trang {page}, dừng.")
            break

        new_found = 0
        for card in cards:
            href = card.get("href", "")
            if not href:
                continue
            full_url = href if href.startswith("http") else BASE_URL + href
            # Lọc chỉ lấy link sản phẩm (không phải danh mục)
            if full_url not in seen and ".html" in full_url and "/mobile" not in full_url:
                seen.add(full_url)
                urls.append(full_url)
                new_found += 1

        log.info(f"  Trang {page}: +{new_found} sản phẩm (tổng: {len(urls)})")

        # Kiểm tra nút "Trang tiếp"
        next_btn = (
            soup.select_one("a.next, a[aria-label='Next'], .pagination .next a")
            or soup.select_one(f"a[href*='page={page+1}']")
        )
        if not next_btn:
            log.info("Đã hết trang danh mục.")
            break

        page += 1

    log.info(f"Tổng {len(urls)} URL sản phẩm tìm được.")
    return urls


# ─── Step 2: Scrape từng sản phẩm ───────────────────────────────────────────
def scrape_product(session: requests.Session, url: str) -> Optional[ProductDTO]:
    soup = fetch(session, url)
    if not soup:
        return None

    dto = ProductDTO(source_url=url, crawled_at=datetime.now().isoformat(timespec="seconds"))

    _parse_name(soup, dto, url)
    _parse_thumbnail(soup, dto)
    _parse_short_description(soup, dto)
    _parse_detail_description(soup, dto)
    _parse_specs(soup, dto)
    _parse_images(soup, dto)
    _parse_variants(soup, dto, session, url)

    return dto


# ─── Parsers ─────────────────────────────────────────────────────────────────
def _parse_name(soup: BeautifulSoup, dto: ProductDTO, url: str):
    h1 = soup.select_one("h1")
    if h1:
        full = h1.get_text(strip=True)
        # Bỏ " | Chính hãng" và tương tự
        full = re.split(r"\s*[|\-–]\s*chính hãng", full, flags=re.IGNORECASE)[0].strip()
    else:
        # Fallback: lấy từ URL
        full = url.split("/")[-1].replace(".html", "").replace("-", " ").title()

    dto.name      = full
    dto.base_name = re.sub(r"\s*\d+\s*(GB|TB)\b", "", full, flags=re.IGNORECASE).strip()
    dto.slug      = generate_slug(dto.base_name)
    dto.brand_slug = detect_brand(full)

    # Series slug = base_name slug
    dto.series_slug = dto.slug


def _parse_thumbnail(soup: BeautifulSoup, dto: ProductDTO):
    selectors = [
        ".product-img img",
        ".gallery-top img",
        ".product-image img",
        "img.product__image",
        ".thumb-product img",
    ]
    for sel in selectors:
        img = soup.select_one(sel)
        if img:
            src = img.get("data-src") or img.get("src") or ""
            if src and "placeholder" not in src:
                dto.thumbnail_url = src
                return


def _parse_short_description(soup: BeautifulSoup, dto: ProductDTO):
    selectors = [
        ".product-feature li",
        ".box-feature li",
        ".feature-list li",
        ".short-description li",
    ]
    for sel in selectors:
        items = soup.select(sel)
        if items:
            lines = [f"- {el.get_text(strip=True)}" for el in items]
            dto.short_description = "\n".join(lines)
            return


def _parse_detail_description(soup: BeautifulSoup, dto: ProductDTO):
    selectors = [
        ".product-description",
        ".detail-content",
        ".box-description",
        "#product-description",
        ".product-info__description",
    ]
    for sel in selectors:
        el = soup.select_one(sel)
        if el:
            dto.detail_description = str(el)
            return


def _parse_specs(soup: BeautifulSoup, dto: ProductDTO):
    """Parse bảng thông số kỹ thuật."""
    # Selector bảng specs
    rows = (
        soup.select(".technical-content tr")
        or soup.select("table.specs tr")
        or soup.select(".box-specs tr")
        or soup.select(".product-params tr")
    )

    order = 0
    for row in rows:
        cells = row.select("td")
        if len(cells) >= 2:
            key   = cells[0].get_text(strip=True)
            value = cells[1].get_text(" ", strip=True)
            if key and value:
                dto.specs.append(SpecDTO(key=key, value=value, sort_order=order))
                order += 1

    # Fallback: dl/dt/dd
    if not dto.specs:
        dts = soup.select(".specs-list dt, .param-list dt")
        dds = soup.select(".specs-list dd, .param-list dd")
        for i, (dt, dd) in enumerate(zip(dts, dds)):
            dto.specs.append(SpecDTO(
                key=dt.get_text(strip=True),
                value=dd.get_text(strip=True),
                sort_order=i
            ))


def _parse_images(soup: BeautifulSoup, dto: ProductDTO):
    """Parse gallery ảnh."""
    selectors = [
        ".product-gallery a",
        ".gallery-thumbs img",
        ".product-image-gallery img",
        ".slick-track img",
        ".thumb-list img",
    ]
    seen = set()
    for sel in selectors:
        els = soup.select(sel)
        for el in els:
            src = (
                el.get("href")
                or el.get("data-src")
                or el.get("src")
                or ""
            )
            # Lấy URL full size từ CDN (bỏ resize prefix)
            src = re.sub(r"https://cdn\d\.cellphones\.com\.vn/\d+x\d*/", 
                         "https://cdn2.cellphones.com.vn/x/", src)
            if src and "cdn" in src and "placeholder" not in src and src not in seen:
                seen.add(src)
                dto.image_urls.append(src)
        if dto.image_urls:
            break


def _parse_price_block(soup: BeautifulSoup) -> tuple[float, float]:
    """Trả về (price, compare_at_price)."""
    # Giá sale (đỏ)
    price_el = (
        soup.select_one(".box-price .tpt--sale")
        or soup.select_one(".product-price .price-sale")
        or soup.select_one(".price-box .price")
        or soup.select_one("[class*='price-sale']")
        or soup.select_one("[class*='price'] strong")
    )
    # Giá gốc gạch ngang
    compare_el = (
        soup.select_one(".box-price .tpt--compare")
        or soup.select_one(".product-price del")
        or soup.select_one("[class*='price-old']")
        or soup.select_one("del")
    )
    price      = clean_price(price_el.get_text()     if price_el   else "")
    compare_at = clean_price(compare_el.get_text()   if compare_el else "")
    return price, compare_at


def _parse_variants(
    soup: BeautifulSoup,
    dto: ProductDTO,
    session: requests.Session,
    current_url: str
):
    """Parse tất cả variants (màu × storage)."""

    # ── Storage options ──────────────────────────────────────────
    storage_els = (
        soup.select(".storage-option a")
        or soup.select(".variant-storage a")
        or soup.select("[class*='storage'] a")
    )
    storage_labels = [el.get_text(strip=True) for el in storage_els if el.get_text(strip=True)]

    # ── Color options ────────────────────────────────────────────
    color_els = (
        soup.select(".color-option a")
        or soup.select(".variant-color a")
        or soup.select("[class*='color'] a[href]")
    )

    # ── Giá trang hiện tại ───────────────────────────────────────
    price, compare_at = _parse_price_block(soup)

    # ── Màu và storage đang active ───────────────────────────────
    active_color = (
        soup.select_one(".color-option a.active, .color-option .active")
        or soup.select_one("[class*='color-selected']")
    )
    active_storage = (
        soup.select_one(".storage-option a.active")
        or soup.select_one("[class*='storage-selected']")
    )

    current_color   = active_color.get("title", "") or (active_color.get_text(strip=True) if active_color else "") 
    current_storage = active_storage.get_text(strip=True) if active_storage else (storage_labels[0] if storage_labels else "")

    if not current_color:
        # Fallback: lấy từ title của color link hiện tại
        for cel in color_els:
            if cel.get("href", "") in current_url or "active" in cel.get("class", []):
                current_color = cel.get("title", "") or cel.get_text(strip=True)
                break

    # Tạo variant cho URL hiện tại
    v = _build_variant(current_color, current_storage, price, compare_at, current_url, dto.base_name)
    dto.variants.append(v)

    # ── Các màu khác: scrape thêm ────────────────────────────────
    scraped_urls = {current_url}

    for cel in color_els:
        href = cel.get("href", "")
        if not href:
            continue
        color_url = href if href.startswith("http") else BASE_URL + href
        if color_url in scraped_urls:
            continue

        color_name    = cel.get("title", "") or cel.get_text(strip=True)
        color_img_url = cel.select_one("img")
        color_img_url = (color_img_url.get("src") or "") if color_img_url else ""
        scraped_urls.add(color_url)

        # Scrape trang màu đó
        color_soup = fetch(session, color_url)
        if not color_soup:
            continue

        cp, cmp_at = _parse_price_block(color_soup)
        cv = _build_variant(color_name, current_storage, cp, cmp_at, color_url, dto.base_name)
        cv.color_image_url = color_img_url
        dto.variants.append(cv)

    # ── Các storage khác ─────────────────────────────────────────
    for sel in storage_els:
        href = sel.get("href", "")
        if not href:
            continue
        storage_url = href if href.startswith("http") else BASE_URL + href
        storage_label = sel.get_text(strip=True)

        if storage_url in scraped_urls or storage_label == current_storage:
            continue
        scraped_urls.add(storage_url)

        s_soup = fetch(session, storage_url)
        if not s_soup:
            continue

        sp, smp_at = _parse_price_block(s_soup)
        sv = _build_variant(current_color, storage_label, sp, smp_at, storage_url, dto.base_name)
        dto.variants.append(sv)

    # Tính sale % chung cho Product (lấy từ variant đầu)
    if dto.variants:
        first = dto.variants[0]
        if first.discount:
            dto.sale = first.discount.discount_percent


def _build_variant(
    color: str,
    storage_label: str,
    price: float,
    compare_at: float,
    url: str,
    base_name: str
) -> VariantDTO:
    slug = url.replace(BASE_URL + "/", "").replace(".html", "")
    v = VariantDTO(
        color           = color,
        storage_label   = storage_label,
        storage_gb      = parse_storage_gb(storage_label),
        price           = price,
        compare_at_price= compare_at if compare_at > price else None,
        slug            = slug,
        sku             = generate_sku(base_name, storage_label, color),
        discount        = make_discount(price, compare_at),
    )
    return v


# ─── Main ─────────────────────────────────────────────────────────────────────
def main():
    session     = make_session()
    all_products = []

    # 1. Lấy danh sách URL sản phẩm
    product_urls = get_product_urls(session)

    if not product_urls:
        log.warning("Không tìm được URL sản phẩm nào!")
        return

    # 2. Scrape từng sản phẩm (song song)
    log.info(f"Bắt đầu scrape {len(product_urls)} sản phẩm...")
    success = 0
    failed  = 0

    # Dùng ThreadPoolExecutor để tăng tốc
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_url = {
            executor.submit(scrape_product, make_session(), url): url
            for url in product_urls
        }

        for future in as_completed(future_to_url):
            url = future_to_url[future]
            try:
                product = future.result()
                if product:
                    all_products.append(asdict(product))
                    success += 1
                    log.info(f"  ✓ [{success}/{len(product_urls)}] {product.name}")
                else:
                    failed += 1
                    log.warning(f"  ✗ Scrape thất bại: {url}")
            except Exception as e:
                failed += 1
                log.error(f"  ✗ Lỗi khi scrape {url}: {e}")

    # 3. Ghi JSON
    output = {
        "crawled_at":    datetime.now().isoformat(timespec="seconds"),
        "total":         len(all_products),
        "source":        MOBILE_URL,
        "products":      all_products,
    }

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    log.info(f"""
╔══════════════════════════════════════╗
║         SCRAPE HOÀN THÀNH           ║
╠══════════════════════════════════════╣
║  Thành công : {success:<24}║
║  Thất bại   : {failed:<24}║
║  Output     : {OUTPUT_FILE:<24}║
╚══════════════════════════════════════╝
""")


if __name__ == "__main__":
    main()

2026-05-23 16:32:35,700 [INFO] Bắt đầu lấy danh sách sản phẩm...
2026-05-23 16:32:35,702 [INFO]   Đang crawl trang danh mục: https://cellphones.com.vn/mobile.html
2026-05-23 16:32:38,820 [INFO]   Trang 1: +20 sản phẩm (tổng: 20)
2026-05-23 16:32:38,903 [INFO] Đã hết trang danh mục.
2026-05-23 16:32:38,903 [INFO] Tổng 20 URL sản phẩm tìm được.
2026-05-23 16:32:38,904 [INFO] Bắt đầu scrape 20 sản phẩm...
2026-05-23 16:32:41,449 [ERROR]   ✗ Lỗi khi scrape https://cellphones.com.vn/iphone-17-pro-max.html: 'NoneType' object has no attribute 'get'
2026-05-23 16:32:41,952 [ERROR]   ✗ Lỗi khi scrape https://cellphones.com.vn/iphone-17-pro.html: 'NoneType' object has no attribute 'get'
2026-05-23 16:32:42,459 [ERROR]   ✗ Lỗi khi scrape https://cellphones.com.vn/dien-thoai-oppo-find-x9s.html: 'NoneType' object has no attribute 'get'
2026-05-23 16:32:44,457 [ERROR]   ✗ Lỗi khi scrape https://cellphones.com.vn/dien-thoai-samsung-galaxy-s26-ultra.html: 'NoneType' object has no attribute 'get'
2026-

In [17]:
print("Done.")

Done.
